In [ ]:
"""Performance stratification of the voice-derived MetaVoX risk score """
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from help.stats_utils import SEED, N_BOOT                      
from help.system_defs import add_system_flags

In [ ]:
MIN_PER_CLASS = 20  # floor on the minority class within a stratum for AUC/Sens/Spec to be
                     # considered stable enough to report; below this, the bootstrap CI is
                     # too wide to be informative and the stratum is flagged "underpowered"

POP_PATH = "analytic_population.pkl"
OOF_PATH = "Results_MoE/oof_predictions.csv"
OUT_DIR = "Results_Cofactor_Stratification"
os.makedirs(OUT_DIR, exist_ok=True)

BINARY_STRATA = {"Smoking": "is_smoker", "Alcohol": "is_current_drinker"}
CONTRASTS = {name: ("Present", "Absent") for name in BINARY_STRATA}
CONTRASTS.update({"Age": ("Oldest", "Youngest")})
METRICS = ["AUC", "Sensitivity", "Specificity"]

In [ ]:
# ---- 1. Load and define strata ----
pop = add_system_flags(pd.read_pickle(POP_PATH))
pop["gender"] = pop["gender"].str.lower().str.strip()
oof = pd.read_csv(OOF_PATH)[["participant_id", "predicted_composite_score", "y_any", "flag_rate_composite"]]
df = pop.drop(columns="y_any", errors="ignore").merge(oof, on="participant_id", how="inner", validate="1:1")
assert len(df) == len(pop) == len(oof), "participants differ between population file and 02_train output"
df["score"] = df["predicted_composite_score"] / 100

for name, col in BINARY_STRATA.items():
    df[name] = df[col].map({1: "Present", 0: "Absent"})
df["Age"] = (df.groupby("gender")["age"].transform(lambda a: pd.qcut(a, 3, labels=False))
               .map({0: "Youngest", 1: "Middle", 2: "Oldest"}))
df["Overall"] = "All"
AXES = [*BINARY_STRATA, "Age", "Overall"]

In [ ]:
# ---- 2. Metrics ----
def evaluate(y, s, f, masks):
    out = {}
    for key, m in masks.items():
        yy, ss, ff = y[m], s[m], f[m]
        if len(np.unique(yy)) < 2:
            continue
        out[key] = {"AUC": roc_auc_score(yy, ss),
                    "Sensitivity": ff[yy == 1].mean(), "Specificity": 1 - ff[yy == 0].mean()}
    return out

In [ ]:
# ---- 3. One bootstrap per sex ----
def analyse_sex(d, sex):
    d = d.reset_index(drop=True)
    y, s, f, n = (d["y_any"].to_numpy().astype(int), d["score"].to_numpy(),
                  d["flag_rate_composite"].to_numpy(float), len(d))
    masks = {(a, lvl): (d[a] == lvl).to_numpy() for a in AXES for lvl in d[a].dropna().unique()}
    small = {k: min(y[m].sum(), (1 - y[m]).sum()) < MIN_PER_CLASS for k, m in masks.items()}
    estimate = evaluate(y, s, f, masks)

    rng = np.random.default_rng(SEED)
    reps = []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n)
        reps.append(evaluate(y[i], s[i], f[i], {k: m[i] for k, m in masks.items()}))

    rows = []
    def add(axis, stratum, metric, est, boot, underpowered, size=np.nan):
        if not boot:
            return
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append(dict(sex=sex, axis=axis, stratum=stratum, n=size, metric=metric, estimate=est,
                         ci_lo=lo, ci_hi=hi, n_valid_boot=len(boot), underpowered=underpowered))

    for (axis, lvl), res in estimate.items():
        for metric in METRICS:
            boot = [r[(axis, lvl)][metric] for r in reps if (axis, lvl) in r]
            add(axis, lvl, metric, res[metric], boot, small[(axis, lvl)], masks[(axis, lvl)].sum())
    for axis, (a, b) in CONTRASTS.items():
        if (axis, a) in estimate and (axis, b) in estimate:
            for metric in METRICS:
                est = estimate[(axis, a)][metric] - estimate[(axis, b)][metric]
                boot = [r[(axis, a)][metric] - r[(axis, b)][metric] for r in reps if (axis, a) in r and (axis, b) in r]
                add(axis, f"{a} - {b}", metric, est, boot, small[(axis, a)] or small[(axis, b)])
    return pd.DataFrame(rows)

results = pd.concat([analyse_sex(df[df["gender"] == g], g) for g in ["female", "male"]], ignore_index=True)
results["ci_excludes_0"] = ((results.ci_lo > 0) | (results.ci_hi < 0)) & results["stratum"].str.contains(" - ")
results.to_csv(f"{OUT_DIR}/stratification_results.csv", index=False)

In [ ]:
# ---- 4. Table: estimate (95% CI), one row per stratum ----
TABLE_GROUPS = [
    ("Demographics", [("Age", ["Youngest", "Middle", "Oldest"])]),
    ("Lifestyle factors and symptoms", [("Smoking", ["Present", "Absent"]),
                                        ("Alcohol", ["Present", "Absent"])]),
]
strat = results[~results.stratum.str.contains(" - ")]
cell = strat.assign(txt=lambda t: t.estimate.map("{:.2f}".format) + " (" + t.ci_lo.map("{:.2f}".format)
                    + "-" + t.ci_hi.map("{:.2f}".format) + ")").set_index(["axis", "stratum", "sex", "metric"])["txt"]
nn = strat.drop_duplicates(["axis", "stratum", "sex"]).set_index(["axis", "stratum", "sex"])["n"]
rows = []
for category, variables in TABLE_GROUPS:
    for axis, levels in variables:
        for lvl in levels:
            r = {"Category": category, "Variable": axis, "Level": lvl}
            for sex in ["female", "male"]:
                r[f"{sex} n"] = int(nn.get((axis, lvl, sex), np.nan)) if (axis, lvl, sex) in nn.index else ""
                for m in METRICS:  # Specificity, Sensitivity, AUC
                    r[f"{sex} {m}"] = cell.get((axis, lvl, sex, m), "--")
            rows.append(r)
table3 = pd.DataFrame(rows)

table3.to_csv(f"{OUT_DIR}/table2_stratification.csv", index=False)
print("\nTable 2. Performance by stratum")
print(table3.to_string(index=False))

In [ ]:
# ---- age tertile boundaries by gender ----
for sex, d in df.groupby("gender"):
    q = pd.qcut(d["age"], 3, labels=["Youngest", "Middle", "Oldest"])
    for lvl in ["Youngest", "Middle", "Oldest"]:
        sub = d.loc[q == lvl, "age"]
        print(f"{sex:>6} | {lvl:<14}: n={len(sub):>4}  range {sub.min():.0f}-{sub.max():.0f} y  "
              f"median {sub.median():.1f}")